# Phase 01 - Command line interface

Argparse CLI for the generated `group3-starvenus.py`. This notebook is exported last (`90_`), so it
may refer to anything defined in the earlier notebooks.

Cells whose first line is `#| export` are copied into the generated file; `tools/export_py.py`
appends the `if __name__ == "__main__": main()` guard. Exported cells must use only the standard
library, numpy, matplotlib, cv2, sklearn, skimage, torch and torchvision, and no notebook syntax
(`!`, `%`, `display(`).

Global flags (`--data-root`, `--device`, `--seed-offset`, `--fast`) are accepted both before and
after the subcommand.

To add a command: define its handler (`def cmd_xxx(args) -> int`) in the phase's notebook and add a `Command(...)`
entry to `COMMANDS` below (`reproduce` chains the commands, see its section).

In [ ]:
#| export
import argparse
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

DEFAULT_DATA_ROOT = "Dhan-Shomadhan"
DEVICE_CHOICES = ("auto", "cuda", "mps", "cpu")
DEFAULT_SPLITS = (0, 1, 2, 3, 4)

EXIT_OK = 0
EXIT_FAILURE = 1

## Experiment registry and `choices.json` (T3.1)

`EXPERIMENTS` holds every row of the README matrix (section 5), keyed by the experiment ID that is used
verbatim in run folders, tables and CLI flags. Rows of later phases exist as entries too; running
them fails with a message that names the phase or the missing choice, never with a traceback.

Three things are decided by data and live in `_work/results/choices.json`, which is written by the
gates of phase 04 (and `choose-lr`). A registry entry leaves the field empty (`None`) and
`resolve_experiment` fills it at run time:

```json
{
  "input_mode": {"value": "rect",          "evidence": {"E0-rect": {"mean": 0.91, "std": 0.02}, "...": {}}},
  "backbone":   {"value": "convnext_tiny", "evidence": {"...": {}}},
  "lr": {
    "resnet50":      {"value": 0.0003, "evidence": {"experiment": "E0-rect", "split": 0,
                                                     "val_mixed_macro_f1": {"1e-4": 0.88, "3e-4": 0.91}}},
    "convnext_tiny": {"value": 0.0001, "evidence": {"...": "..."}},
    "swin_t":        {"value": 0.0001, "evidence": {"...": "..."}}
  },
  "composite_n_factor": {"value": 1.0, "evidence": {"...": "..."}},
  "recommended": {"value": "E3-A", "evidence": {}}
}
```

* `input_mode` - gate G0, the winner of E0 ("best of E0" in the matrix). One of `squash`, `pad`, `rect`.
* `backbone` - gate G1, the winner among ResNet50 / ConvNeXt-T / Swin-T ("best of E2").
* `lr` - learning rate **per backbone**, chosen by validation mixed macro-F1 from the split-0 sweep over
  {1e-4, 3e-4}. Written by `choose-lr --backbone NAME`. The learning rate of the winning backbone is
  `lr[backbone]`; there is no separate top-level lr.
* `composite_n_factor` - composites per epoch of Method A as a multiple of the white training count, 0.5 or 1.0, written by
  `choose-composites` from the split-0 sweep (`run --exp E3-A --sweep-composites`). Optional: without it the default 1x is used.
  It is not a field of `ExperimentConfig`; it enters the fingerprint of `use_composites` rows only (see `30_copy_paste.ipynb`).
* `recommended` - gate G2 (phase 06), the experiment ID that `predict` uses by default.

**Aliases (one training, several names).** `E1-U` is the winning E0 row. `E1-WW` is `E1-WF`: both train on white
and pick the checkpoint on white val, so they are the same model, and the `E1-WF` metrics already contain the white
test. `X-domain-WW` is the W->F row of the winning backbone (`E1-WF`, `E2-cnx-WF` or `E2-swin-WF`). Aliases resolve to
the target's config and run folder (`FIXED_ALIASES`, `BACKBONE_ALIASES`), so `run` skips them once the target is done and
`summarize` still shows an `E1-WW` row (white test from the target's runs). `X-domain-FF` has no counterpart for
ConvNeXt-T / Swin-T, so it is a real run.

**Frozen decisions.** `FROZEN_CHOICES` (embedded in the `.py` from the committed `choices.json`, see `tools/export_py.py`)
holds the decisions frozen at `results-freeze` with their evidence. `prepare` writes it to `choices.json` when that file does not
exist yet (`seed_choices`; an existing file is never touched, `load_choices` has no side effects). So with only the `.py`,
`download-weights` and `prepare`, `run --exp E2-swin-U` trains exactly the reported configuration (same config fingerprint).

**Gates and choosers verify once a decision exists** (`settle_choice`). `gate g0`, `gate g1`, `choose-lr` and `choose-composites`
still recompute their decision from the runs, then
* no entry yet: write it with its evidence;
* an entry with the same value: keep the recorded entry (the evidence is not rewritten), print that the decisions match, with the
  recomputed numbers next to the recorded ones;
* an entry with another value: keep the recorded entry, print a warning and exit 0 (so that `reproduce` continues with the
  reported configuration); `--force` overwrites it with the recomputed decision.

Each entry may be written as `{"value": ..., "evidence": ...}` (what the gates write) or as the bare value
(`"input_mode": "rect"`, `"lr": {"resnet50": 0.0003}`), which is convenient by hand. A key that is missing,
or has an invalid value, gives a `ChoicesError` that says which gate or command writes it.

In [ ]:
#| export
import difflib

CHOICES_FILENAME = "choices.json"
SWEEP_LRS = (1e-4, 3e-4)  # T3.4: split-0 learning-rate sweep
# Experiment whose split-0 run is swept for each backbone (04: ResNet50 on rect, the others on their U row).
SWEEP_EXPERIMENT = {"resnet50": "E0-rect", "convnext_tiny": "E2-cnx-U", "swin_t": "E2-swin-U"}
# Used by `--fast` when choices.json does not exist yet, so that every experiment ID can be sanity-checked.
FAST_PLACEHOLDERS = {"input_mode": "rect", "backbone": "resnet50", "lr": 3e-4}
CHOICE_SOURCES = {
    "input_mode": 'the winning E0 input mode ("best of E0"), written by gate G0 (phase 04, Batch 1)',
    "backbone": 'the winning backbone ("best of E2"), written by gate G1 (phase 04, Batch 3)',
    "lr": "the learning rate of this backbone, written by `choose-lr --backbone NAME` after the split-0 sweep "
    "(`run --exp <ID> --sweep-lr`)",
    "composite_n_factor": "the composites per epoch of Method A (x the white training count), written by `choose-composites` "
    "after the split-0 comparison (`run --exp E3-A --sweep-composites`)",
    "recommended": "the recommended method's experiment ID (plain G1 unified row, E3-A or E4-B), written and frozen by gate G2 "
    "(`gate g2`, phase 06)",
}


def choices_path(data_root) -> Path:
    return results_dir(data_root) / CHOICES_FILENAME


@dataclass(frozen=True)
class Choices:
    """The contents of choices.json plus its path (for error messages)."""

    path: Path
    data: dict

    def raw(self, key: str, sub: str | None = None):
        entry = self.data.get(key)
        if sub is not None:
            entry = entry.get(sub) if isinstance(entry, dict) else None
        if isinstance(entry, dict) and "value" in entry:
            entry = entry["value"]
        return entry

    def evidence(self, key: str, sub: str | None = None):
        """The evidence stored next to a choice, or None (no entry, or a bare value)."""
        entry = self.data.get(key)
        if sub is not None:
            entry = entry.get(sub) if isinstance(entry, dict) else None
        return entry.get("evidence") if isinstance(entry, dict) and "value" in entry else None

    def get(self, key: str, sub: str | None = None):
        """The validated choice, or ChoicesError that says which step writes it."""
        value = self.raw(key, sub)
        label = key if sub is None else f"{key}[{sub}]"
        if value is None:
            where = f"{self.path} does not exist" if not self.path.is_file() else f"{self.path} has no such entry"
            raise ChoicesError(f"{where} (needed: {label}). It holds {CHOICE_SOURCES[key]}. Schema: see 90_cli.ipynb.")
        valid = {
            "input_mode": lambda v: v in INPUT_MODES,
            "backbone": lambda v: v in BACKBONES,
            "lr": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool) and 0 < v < 1,
            "composite_n_factor": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool) and float(v) in COMPOSITE_N_FACTORS,
            "recommended": lambda v: v in EXPERIMENTS,
        }[key]
        if not valid(value):
            options = {"input_mode": INPUT_MODES, "backbone": BACKBONES, "lr": "a number in (0, 1)",
                       "composite_n_factor": f"one of {COMPOSITE_N_FACTORS}", "recommended": "an experiment ID"}[key]
            raise ChoicesError(f"{self.path}: invalid {label} = {value!r} (expected {options})")
        return float(value) if key in ("lr", "composite_n_factor") else value


def load_choices(data_root) -> Choices:
    path = choices_path(data_root)
    if not path.is_file():
        return Choices(path, {})
    try:
        data = json.loads(read_file_bytes(path).decode("utf-8"))
    except ValueError as exc:
        raise ChoicesError(f"{path} is not valid JSON: {exc}") from exc
    if not isinstance(data, dict):
        raise ChoicesError(f"{path}: expected a JSON object at the top level")
    return Choices(path, data)


def write_choice(data_root, key: str, value, evidence=None, *, sub: str | None = None) -> Path:
    """Set choices[key] (or choices[key][sub]) to {"value": value, "evidence": evidence}, keeping the rest."""
    choices = load_choices(data_root)
    data = dict(choices.data)
    entry = {"value": value, "evidence": evidence if evidence is not None else {}}
    if sub is None:
        data[key] = entry
    else:
        group = dict(data[key]) if isinstance(data.get(key), dict) else {}
        group[sub] = entry
        data[key] = group
    write_json(choices.path, data, indent=2)
    return choices.path


def seed_choices(data_root, out=print) -> bool:
    """`prepare` step: write choices.json from FROZEN_CHOICES (the decisions frozen at results-freeze, embedded in this
    file) when it does not exist yet. An existing file is never touched. Returns True when the file was written."""
    path = choices_path(data_root)
    if path.is_file():
        return False
    write_json(path, FROZEN_CHOICES, indent=2)
    out(f"prepare: seeded {path} from the decisions frozen at results-freeze and embedded in this file "
        f"(recommended model: {FROZEN_CHOICES['recommended']['value']})")
    return True


def same_choice(recorded, value) -> bool:
    """Equal choice values; numbers (learning rates, composite factors) compare with a tolerance."""
    numbers = (int, float)
    if isinstance(recorded, numbers) and isinstance(value, numbers) and not isinstance(recorded, bool) and not isinstance(value, bool):
        return abs(float(recorded) - float(value)) <= 1e-12
    return recorded == value


def settle_choice(data_root, key: str, value, evidence, *, sub: str | None = None, force: bool = False,
                  command: str, label: str, compare=None, out=print) -> str:
    """Record a decision recomputed from the runs, or verify it against the recorded one (see the section text).

    Returns "written" (no entry yet), "matches" (same value: the recorded entry and its evidence are kept), "differs" (other
    value: the recorded entry is kept and a warning is printed) or "overwritten" (``force``). ``compare(recorded evidence,
    new evidence)`` gives the lines that put the recorded and the recomputed numbers side by side."""
    choices = load_choices(data_root)
    recorded = choices.raw(key, sub)
    if recorded is None:
        write_choice(data_root, key, value, evidence, sub=sub)
        return "written"
    if force:
        write_choice(data_root, key, value, evidence, sub=sub)
        return "overwritten"
    numbers = list(compare(choices.evidence(key, sub), evidence)) if compare is not None else []
    if same_choice(recorded, value):
        out(f"{command}: the recomputed {label} = {value!r} matches the recorded one; the recorded entry and its evidence in "
            f"{choices.path} are kept")
    else:
        out(f"{command}: WARNING - the recomputed {label} = {value!r} differs from the recorded {recorded!r}. The recorded one is "
            "kept: it was frozen at results-freeze and the later experiments use it, so that the tables you reproduce are the "
            "reported ones. --force overwrites it with the recomputed one")
    for line in numbers:
        out(line)
    return "matches" if same_choice(recorded, value) else "differs"


def sweep_comparison(recorded, new: dict, name) -> list[str]:
    """Lines that put the recorded and the recomputed validation mixed macro-F1 of the sweep runs side by side (``name`` turns
    a sweep tag such as '1e-4' into its label)."""
    was = recorded.get("val_mixed_macro_f1") if isinstance(recorded, dict) else None
    lines = ["  val mixed macro-F1 of the split-0 sweep runs:"]
    for tag, score in new["val_mixed_macro_f1"].items():
        before = (was or {}).get(tag)
        lines.append(f"    {name(tag):<10}recorded {fmt3(before) if isinstance(before, (int, float)) else 'not recorded':<13}recomputed {fmt3(score)}")
    return lines


def _row(exp_id: str, backbone, train, test_subsets, **kwargs) -> ExperimentConfig:
    train = tuple(train)
    return ExperimentConfig(
        exp_id=exp_id,
        backbone=backbone,
        train_domains=train,
        select_domains=train,
        test_subsets=tuple(test_subsets),
        input_mode=kwargs.pop("input_mode", None),
        **kwargs,
    )


_U, _W, _F = DOMAINS, ("white",), ("field",)
_ALL3 = ("white", "field", "mixed")
_WF_TEST = ("field", "white")  # field is the main result of a W->F row, white the reference

_ROWS = (
    # E0: input-size ablation, ResNet50 on white + field, equal pixel budget.
    _row("E0-squash", "resnet50", _U, _ALL3, input_mode="squash", note="320x320, aspect distorted"),
    _row("E0-pad", "resnet50", _U, _ALL3, input_mode="pad", note="320x320 letterbox"),
    _row("E0-rect", "resnet50", _U, _ALL3, input_mode="rect", note="448x224 (HxW)"),
    # E1: ResNet50 baseline and domain experiments at the winning E0 input mode.
    _row("E1-U", "resnet50", _U, _ALL3, alias_of_winner="E0", note="alias of the winning E0 row, no retraining"),
    _row("E1-WW", "resnet50", _W, ("white",), note="train white, test white"),
    _row("E1-FF", "resnet50", _F, ("field",), note="train field, test field"),
    _row("E1-WF", "resnet50", _W, _WF_TEST, note="train white, test field (cross-domain)"),
    # E2: ConvNeXt-T and Swin-T.
    _row("E2-cnx-U", "convnext_tiny", _U, _ALL3),
    _row("E2-cnx-WF", "convnext_tiny", _W, _WF_TEST),
    _row("E2-swin-U", "swin_t", _U, _ALL3),
    _row("E2-swin-WF", "swin_t", _W, _WF_TEST),
    # Later phases: runnable only once their hooks exist.
    _row("E3-A", None, _U, _ALL3, use_composites=True, note="Method A (phase 05): composites as extra training samples"),
    _row("E3-A-WF", None, _W, _WF_TEST, use_composites=True, note="white + composites; field images are only backgrounds"),
    _row("S-seg", "lraspp_mobilenet_v3_large", _U, (), kind="segmenter", epochs=SEG_EPOCHS, lr=SEG_LR, weight_decay=SEG_WEIGHT_DECAY,
         batch_size=SEG_BATCH_SIZE, stage1_epochs=0, warmup_epochs=0, label_smoothing=0.0, dropout=0.0, clip_norm=0.0, patience=0,
         note="leaf segmenter (phase 06): LRASPP-MobileNetV3, composites + white pseudo-masks, 512x256"),
    _row("E4-B", None, _U, _ALL3, seg_crop=True, note="Method B (phase 06): crop to the S-seg box (stored boxes.json), classify"),
    _row("X-domain-WW", None, _W, ("white",), note="best backbone, train white, test white"),
    _row("X-domain-FF", None, _F, ("field",), note="best backbone, train field, test field"),
    _row("X-gradcam", None, _U, (), kind="analysis", note="Grad-CAM shortcut analysis (phase 07), see the gradcam command"),
)
EXPERIMENTS: dict[str, ExperimentConfig] = {row.exp_id: row for row in _ROWS}
# `run --exp <GROUP>` runs every member in order.
EXPERIMENT_GROUPS = {
    "E0": ("E0-squash", "E0-pad", "E0-rect"),
    "E1": ("E1-U", "E1-WW", "E1-FF", "E1-WF"),
    "E2": ("E2-cnx-U", "E2-cnx-WF", "E2-swin-U", "E2-swin-WF"),
    "X-domain": ("X-domain-WW", "X-domain-FF"),
}
# Rows that are the same training as another row (same data, selection domains, backbone, input mode, lr and
# switches; they differ only in which test subset the report highlights). They have no run folder of their own:
# like E1-U they resolve to the target's config and folder, so nothing is trained twice. Aliases do not change
# any ExperimentConfig, so no existing fingerprint moves.
#   E1-WW: train white, select on white, same as E1-WF (whose metrics also hold the white test as reference).
FIXED_ALIASES = {"E1-WW": "E1-WF"}
#   X-domain-WW (best backbone, train white): the W->F row of the winning backbone is the same training.
#   X-domain-FF has no counterpart for ConvNeXt-T / Swin-T (there is no E2 F->F row), so it is not aliased.
BACKBONE_ALIASES = {"X-domain-WW": {"resnet50": "E1-WF", "convnext_tiny": "E2-cnx-WF", "swin_t": "E2-swin-WF"}}
NON_CLASSIFIER_MESSAGE = {
    "segmenter": "is the leaf segmenter, run with `run --exp S-seg` (it is not a classifier)",
    "analysis": "is a Grad-CAM analysis, produced by the `gradcam` command of phase 07 (07-analysis.md)",
}


def run_folder_name(exp_id: str, choices: Choices) -> str:
    """Folder (under runs/) that holds the results of an experiment ID: itself, except for the E1-U alias."""
    if exp_id not in EXPERIMENTS:
        expand_experiment_ids(exp_id)  # raises with suggestions
    if exp_id in FIXED_ALIASES:
        return run_folder_name(FIXED_ALIASES[exp_id], choices)
    if exp_id in BACKBONE_ALIASES:
        return run_folder_name(BACKBONE_ALIASES[exp_id][choices.get("backbone")], choices)
    base = EXPERIMENTS[exp_id]
    return f"{base.alias_of_winner}-{choices.get('input_mode')}" if base.alias_of_winner else exp_id


def expand_experiment_ids(text: str) -> list[str]:
    """'E0' -> its three rows; 'E1-U,E1-WW' -> two rows. Unknown IDs raise UnknownExperimentError."""
    ids: list[str] = []
    for part in (p.strip() for p in text.split(",") if p.strip()):
        for exp_id in EXPERIMENT_GROUPS.get(part, (part,)):
            if exp_id not in EXPERIMENTS:
                close = difflib.get_close_matches(exp_id, [*EXPERIMENTS, *EXPERIMENT_GROUPS], n=3)
                hint = f" Did you mean: {', '.join(close)}?" if close else ""
                raise UnknownExperimentError(
                    f"unknown experiment {exp_id!r}.{hint} Known: {', '.join([*EXPERIMENTS, *EXPERIMENT_GROUPS])}"
                )
            if exp_id not in ids:
                ids.append(exp_id)
    if not ids:
        raise UnknownExperimentError("no experiment ID given")
    return ids


@dataclass(frozen=True)
class ResolvedExperiment:
    config: ExperimentConfig  # fully concrete: backbone, input_mode and lr are set
    requested: str  # the ID that was asked for (differs from config.exp_id for the E1-U alias)
    choices_used: dict  # which values came from choices.json (or FAST placeholders), for config.json
    notes: tuple[str, ...] = ()
    composite_factor: float | None = None  # Method A rows: composites per epoch x the white training count; else None


def resolve_experiment(
    exp_id: str, choices: Choices, *, lr: float | None = None, placeholders: bool = False,
    composite_factor: float | None = None,
) -> ResolvedExperiment:
    """Turn a registry entry into a concrete config.

    ``lr`` overrides the choices.json learning rate (sweeps, ``--lr``). ``placeholders=True`` (``--fast``)
    substitutes FAST_PLACEHOLDERS for choices that do not exist yet, and says so in ``notes``. ``composite_factor``
    overrides the chosen composites-per-epoch factor of Method A rows (the split-0 comparison); without it
    `composite_n_factor` of choices.json is used, and the default 1x when `choose-composites` has not run yet.
    """
    if exp_id not in EXPERIMENTS:
        expand_experiment_ids(exp_id)  # raises with a helpful message
    base = EXPERIMENTS[exp_id]
    if base.kind == "segmenter":  # needs no choice: the architecture, lr and recipe are fixed in the registry row
        return ResolvedExperiment(base, exp_id, {}, ())
    if base.kind != "classifier":
        raise NotReadyError(f"{exp_id} {NON_CLASSIFIER_MESSAGE[base.kind]}; it cannot be run as a classifier")
    used: dict = {}
    notes: list[str] = []

    def pick(key: str, sub: str | None = None):
        label = key if sub is None else f"{key}[{sub}]"
        try:
            value = choices.get(key, sub)
            used[label] = value
            return value
        except ChoicesError as exc:
            if not placeholders:
                raise ChoicesError(f"{exp_id}: {exc}") from None
            value = FAST_PLACEHOLDERS[key]
            used[label] = value
            notes.append(f"--fast: no choice for {label}, using the placeholder {value!r}")
            return value

    if exp_id in FIXED_ALIASES or exp_id in BACKBONE_ALIASES:
        target = FIXED_ALIASES[exp_id] if exp_id in FIXED_ALIASES else BACKBONE_ALIASES[exp_id][pick("backbone")]
        inner = resolve_experiment(target, choices, lr=lr, placeholders=placeholders, composite_factor=composite_factor)
        note = f"{exp_id} is an alias of {inner.config.exp_id} (identical training): same run folder, no retraining"
        return ResolvedExperiment(inner.config, exp_id, {**inner.choices_used, **used}, (note, *inner.notes, *notes),
                                  inner.composite_factor)

    if base.alias_of_winner:
        mode = pick("input_mode")
        inner = resolve_experiment(f"{base.alias_of_winner}-{mode}", choices, lr=lr, placeholders=placeholders)
        note = f"{exp_id} is an alias of {inner.config.exp_id} (the winning input mode): same run folder, no retraining"
        return ResolvedExperiment(inner.config, exp_id, {**inner.choices_used, **used}, (note, *inner.notes, *notes),
                                  inner.composite_factor)

    backbone = base.backbone if base.backbone is not None else pick("backbone")
    mode = base.input_mode if base.input_mode is not None else pick("input_mode")
    if lr is None:
        lr = base.lr if base.lr is not None else pick("lr", backbone)
    config = replace(base, backbone=backbone, input_mode=mode, lr=lr)
    factor = None
    if config.use_composites:
        if composite_factor is not None:
            factor, used["composite_n_factor"] = float(composite_factor), float(composite_factor)
        elif choices.raw("composite_n_factor") is not None:
            factor = choices.get("composite_n_factor")
            used["composite_n_factor"] = factor
        else:
            factor = COMPOSITE_DEFAULT_FACTOR
            used["composite_n_factor"] = factor
            notes.append(f"composite_n_factor is not chosen yet (`choose-composites`): using the default {COMPOSITE_DEFAULT_FACTOR:g}x")
    return ResolvedExperiment(config, exp_id, used, tuple(notes), factor)

In [ ]:
#| export
def cmd_download_weights(args: argparse.Namespace) -> int:
    try:
        download_weights(Path(args.data_root))
    except (ExperimentError, OSError) as exc:
        print(f"download-weights: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

## `run`, `choose-lr`, `predict`, `download-weights` (phase 03)

`run --exp ID` trains and evaluates one experiment (or a group: `E0`, `E1`, `E2`, `X-domain`, or a
comma-separated list) on the given splits and writes `_work/runs/<ID>/split_<k>/` (see `20_classifier.ipynb`).
It is idempotent: finished splits are skipped unless `--force` is given.

**Where results go.** Only a plain `run` writes to `_work/runs/<ID>/`, the folder that `summarize` reads.
Anything that is not the real recipe goes elsewhere and can never be mistaken for a result:

| Flag | Output folder |
|---|---|
| `--sweep-lr` (split 0 only, lr 1e-4 and 3e-4) | `_work/runs/sweep/<ID>_lr<lr>/split_0/` |
| `--sweep-composites` (`--exp E3-A`, split 0, 0.5x and 1x composites per epoch) | `_work/runs/sweep/E3-A_n0.5/split_0/`, `E3-A_n1/split_0/` |
| `--fast`, `--lr`, `--epochs`, `--stage1-epochs` (dev overrides) | `_work/runs/scratch/<ID>_fast/` or `<ID>_dev/` (gitignored) |
| `--out-dir DIR` | `DIR/<ID>/` (replaces `_work/runs`) |

`--fast` runs the experiment at smoke scale (24 images per split part, 1 epoch per stage, input size divided by
4, batch 8, real weights only if already downloaded). When `choices.json` does not exist yet it substitutes
placeholder choices and says so, so every experiment ID can be sanity-checked locally in about a minute.
`--batch-size 16 --grad-accum 2` is the manual form of the out-of-memory fallback that `run` applies by itself.

`choose-lr --backbone NAME` implements the T3.4 rule: it reads the two split-0 sweep runs of that backbone,
picks the lr with the higher **validation mixed macro-F1** (ties go to the smaller lr) and records it in
`choices.json` under `lr[NAME]`. It refuses `--fast` and dev-override runs. When `choices.json` already holds an lr for that
backbone (`prepare` seeds the frozen one), the recomputed lr is only verified against it (see above; `--force` overwrites).

`choose-composites` implements the same rule for Method A: it reads the two split-0 sweep runs of `E3-A`, picks the factor with the
higher **validation** mixed macro-F1 (ties go to 1x) and records `composite_n_factor` in `choices.json`; the winning sweep run is copied to
split 0 of `_work/runs/E3-A/` like in `choose-lr`. It verifies against a recorded factor in the same way.

**Sweep reuse.** The sweep run at the winning lr is the same training as split 0 of the real run (same config
fingerprint, same split). So `choose-lr` copies it to `_work/runs/<run folder>/split_0/` (plus a `provenance.json`
that says where it came from) and the real `run` then skips split 0. The sweep copy stays in `sweep/` as the evidence
trail. The copy is refused (with the reason printed) when the fingerprint or split sha256 differs, and a split 0 that
already exists in the real folder is never touched. If the recomputed lr differs from the recorded one, the recorded lr stays in
force and it is the sweep run at the **recorded** lr that is copied (it is the same training as the real split 0; the run at the
other lr never is).

In [ ]:
#| export
FAST_CONFIG = {"stage1_epochs": 1, "epochs": 1, "size_div": 4, "batch_size": 8}
FAST_IMAGES = 24


def pretrained_available(backbone: str, data_root) -> bool:
    """True if the pretrained file is already stored under <data_root>/_work/weights (no download is triggered)."""
    previous = os.environ.get("TORCH_HOME")
    try:
        set_torch_home(data_root, create=False)
        weights = backbone_weights(backbone) if backbone in BACKBONE_SPECS else torchvision.models.get_weight(SEG_ARCHITECTURES[backbone])
        return weight_file_ok(weight_checkpoint_path(weights))
    finally:
        if previous is None:
            os.environ.pop("TORCH_HOME", None)
        else:
            os.environ["TORCH_HOME"] = previous


def run_destination(data_root, args, cfg: ExperimentConfig, dev: bool, composite_factor: float | None = None) -> tuple[Path, str]:
    """(root folder, run name) for one resolved config; see the table in the notebook."""
    sweep_composites = getattr(args, "sweep_composites", False)
    if args.out_dir:
        root = Path(args.out_dir)
    elif dev:
        root = scratch_dir(data_root)
    else:
        root = sweep_dir(data_root) if args.sweep_lr or sweep_composites else runs_dir(data_root)
    if args.sweep_lr:
        return root, sweep_run_name(cfg.exp_id, cfg.lr)
    if sweep_composites:
        return root, composite_sweep_run_name(cfg.exp_id, composite_factor)
    suffix = "" if args.out_dir or not dev else "_fast" if args.fast else "_dev"
    return root, cfg.exp_id + suffix


def apply_run_flags(args, resolved: ResolvedExperiment, data_root) -> tuple[ResolvedExperiment, dict, int | None]:
    """Apply --fast and the dev/system overrides to a resolved experiment. Returns (resolved, overrides, limit)."""
    changes: dict = {}
    overrides: dict = {}
    limit = None
    if args.fast:
        changes.update(FAST_CONFIG)
        changes["pretrained"] = pretrained_available(getattr(args, "seg_arch", None) or resolved.config.backbone, data_root)
        overrides["fast"] = True
        limit = FAST_IMAGES
    for flag, field_name in (("epochs", "epochs"), ("stage1_epochs", "stage1_epochs"), ("lr", "lr")):
        value = getattr(args, flag, None)
        if value is not None:
            changes[field_name] = value
            overrides[flag] = value
    seg_arch = getattr(args, "seg_arch", None)
    if seg_arch is not None:
        if resolved.config.kind != "segmenter":
            raise ExperimentError(f"--seg-arch only applies to the segmenter (S-seg), not {resolved.requested}")
        changes["backbone"] = seg_arch
        overrides["seg_arch"] = seg_arch
    for flag in ("batch_size", "grad_accum"):
        value = getattr(args, flag, None)
        if value is not None:
            changes[flag] = value
    notes = resolved.notes
    if args.fast and not changes["pretrained"]:
        notes = (*notes, f"--fast: {resolved.config.backbone} weights are not downloaded, using random initialisation")
    return replace(resolved, config=replace(resolved.config, **changes), notes=notes), overrides, limit


def run_command(args) -> int:
    data_root = Path(args.data_root)
    choices = load_choices(data_root)
    device = resolve_device(args.device)
    ids = expand_experiment_ids(args.exp)
    sweep = args.sweep_lr
    sweep_composites = getattr(args, "sweep_composites", False)
    if sweep and sweep_composites:
        raise ExperimentError("--sweep-lr and --sweep-composites are two different sweeps; run them one at a time")
    if sweep and args.lr is not None:
        raise ExperimentError("--sweep-lr already sets the learning rates (1e-4 and 3e-4); do not combine it with --lr")
    if sweep_composites and ids != [COMPOSITE_SWEEP_EXPERIMENT]:
        raise ExperimentError(f"--sweep-composites compares the composites per epoch of {COMPOSITE_SWEEP_EXPERIMENT}: use --exp {COMPOSITE_SWEEP_EXPERIMENT}")
    splits = (0,) if sweep or sweep_composites else tuple(args.splits)
    if (sweep or sweep_composites) and tuple(args.splits) != DEFAULT_SPLITS and tuple(args.splits) != (0,):
        print("run: note - the sweeps always use split 0 only")
    dev = bool(args.fast or args.lr is not None or args.epochs is not None or args.stage1_epochs is not None
               or getattr(args, "seg_arch", None) is not None)
    options_base = dict(device=device, seed_offset=args.seed_offset, num_workers=args.num_workers, force=args.force,
                        weights_root=data_root, seg_dir=getattr(args, "seg_dir", None))
    weights_home = set_torch_home(data_root)
    print(f"run: device {device}, data root {data_root}, TORCH_HOME {weights_home}")
    summary, seg_summary = [], []
    for exp_id in ids:
        if sweep:
            variants = [(lr, None) for lr in SWEEP_LRS]  # (learning rate, composites factor) of each run
        elif sweep_composites:
            variants = [(None, f) for f in COMPOSITE_N_FACTORS]
        else:
            variants = [(None, None)]
        for sweep_lr, factor in variants:
            resolved = resolve_experiment(exp_id, choices, lr=sweep_lr if sweep else args.lr, placeholders=args.fast,
                                          composite_factor=factor)
            resolved, overrides, limit = apply_run_flags(args, resolved, data_root)
            cfg = resolved.config
            if sweep and cfg.exp_id != SWEEP_EXPERIMENT.get(cfg.backbone):
                print(f"run: note - choose-lr reads the sweep of {SWEEP_EXPERIMENT.get(cfg.backbone)} for {cfg.backbone}, not {cfg.exp_id}")
            root, name = run_destination(data_root, args, cfg, dev, resolved.composite_factor)
            for note in resolved.notes:
                print(f"run: {note}")
            print(f"run: {resolved.requested} -> {root / name}/split_<k>/"
                  + ("  (sweep, never a result)" if sweep or sweep_composites else "  (scratch, never a result)" if dev else ""))
            options = RunOptions(limit=limit, overrides=tuple(overrides.items()), **options_base)
            for split_seed in splits:
                runner = run_seg_split if cfg.kind == "segmenter" else run_split
                result = runner(resolved, split_seed, data_root=data_root, out_dir=split_dir(root / name, split_seed),
                                options=options, log=lambda text: print(text, flush=True))
                metrics = result["metrics"]
                if cfg.kind == "segmenter":
                    seg_summary.append((name, split_seed, result["status"], metrics["val_iou_composite"], metrics["val_iou_white"]))
                else:
                    summary.append((name, split_seed, result["status"], metrics["val"]["mixed"]["macro_f1"], metrics["test"]["mixed"]["macro_f1"]))
    if summary:
        print("run: summary (macro-F1)")
        print(f"  {'run':<22}{'split':>5}  {'status':<8}{'val mixed':>10}{'test mixed':>11}")
        for name, split_seed, status, val_f1, test_f1 in summary:
            print(f"  {name:<22}{split_seed:>5}  {status:<8}{fmt3(val_f1):>10}{fmt3(test_f1):>11}")
    if seg_summary:
        print("run: summary (segmenter, IoU on validation images)")
        print(f"  {'run':<22}{'split':>5}  {'status':<8}{'composites':>11}{'white':>8}")
        for name, split_seed, status, composite_iou, white_iou in seg_summary:
            print(f"  {name:<22}{split_seed:>5}  {status:<8}{fmt3(composite_iou):>11}{fmt3(white_iou):>8}")
    return EXIT_OK


def cmd_run(args: argparse.Namespace) -> int:
    try:
        return run_command(args)
    except (ExperimentError, DatasetError) as exc:
        print(f"run: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE


def choose_lr(data_root, backbone: str, out=print, *, force: bool = False) -> dict:
    """T3.4 rule: of the two split-0 sweep runs, take the lr with the higher validation mixed macro-F1. The decision is
    recorded in choices.json, or verified against the recorded one when there is one (``settle_choice``; ``force`` overwrites)."""
    if backbone not in SWEEP_EXPERIMENT:
        raise ExperimentError(f"unknown backbone {backbone!r}, expected one of {', '.join(SWEEP_EXPERIMENT)}")
    exp_id = SWEEP_EXPERIMENT[backbone]
    scores, shas = {}, set()
    for lr in SWEEP_LRS:
        run_dir = split_dir(sweep_dir(data_root) / sweep_run_name(exp_id, lr), 0)
        metrics = read_metrics(run_dir)
        if metrics is None:
            raise ExperimentError(f"missing sweep result {run_dir / 'metrics.json'}; run `run --exp {exp_id} --sweep-lr` first")
        if metrics.get("fast") or metrics.get("overrides"):
            raise ExperimentError(f"{run_dir} is a --fast / dev-override run, not a real sweep run; re-run the sweep without those flags")
        if metrics.get("backbone") != backbone or abs(metrics.get("lr", -1) - lr) > 1e-12:
            raise ExperimentError(f"{run_dir}: expected backbone {backbone} at lr {lr}, found {metrics.get('backbone')} at {metrics.get('lr')}")
        scores[lr] = metrics["val"]["mixed"]["macro_f1"]
        shas.add(metrics["split_sha256"])
    if len(shas) != 1:
        raise ExperimentError("the two sweep runs used different splits; re-run both")
    best = max(SWEEP_LRS, key=lambda lr: (scores[lr], -lr))  # ties: the smaller lr
    evidence = {
        "experiment": exp_id, "split": 0, "split_sha256": shas.pop(),
        "val_mixed_macro_f1": {lr_tag(lr): scores[lr] for lr in SWEEP_LRS},
        "rule": "higher validation mixed macro-F1 on split 0; ties go to the smaller lr",
    }
    out(f"choose-lr: {backbone}: " + ", ".join(f"lr {lr_tag(lr)} -> val mixed F1 {fmt3(scores[lr])}" for lr in SWEEP_LRS)
        + f"  =>  lr {lr_tag(best)}")
    outcome = settle_choice(data_root, "lr", best, evidence, sub=backbone, force=force, command="choose-lr", label=f"lr[{backbone}]",
                            compare=lambda recorded, new: sweep_comparison(recorded, new, lambda tag: f"lr {tag}"), out=out)
    if outcome in ("written", "overwritten"):
        out(f"choose-lr: lr[{backbone}] = {lr_tag(best)} written to {choices_path(data_root)}"
            + (" (--force: it replaced the recorded one)" if outcome == "overwritten" else ""))
    chosen = load_choices(data_root).get("lr", backbone)  # in force: the recomputed lr, or the recorded one that was kept
    try:
        reuse = reuse_sweep_run(data_root, backbone, chosen)  # the sweep run at the lr in force is the same training as split 0
        message = {
            "copied": f"the sweep run at lr {lr_tag(chosen)}{' (the recorded lr)' if outcome == 'differs' else ''} was copied to split 0 of "
            f"{run_folder_name(exp_id, load_choices(data_root))} (same training; the real run will skip split 0)",
            "exists": "the real split 0 already exists and was left untouched",
        }[reuse]
    except (ExperimentError, DatasetError) as exc:
        reuse, message = "not reused", f"sweep run not reused for split 0 ({exc}); the real run will train it"
    out(f"choose-lr: {message}")
    return {"lr": chosen, "recomputed_lr": best, "evidence": evidence, "split_0": reuse, "outcome": outcome}


PROVENANCE_FILENAME = "provenance.json"


def reuse_sweep_run(data_root, backbone: str, lr: float) -> str:
    """Copy the sweep run at ``lr`` to split 0 of the real run folder of the backbone's sweep experiment.

    Returns "copied" or "exists" (a real split 0 is already there and is left untouched). Raises ExperimentError
    when the sweep run is not the same training as the real run: different config fingerprint (including a changed
    lr in choices.json), different split sha256, or a --fast / dev-override run."""
    data_root = Path(data_root)
    exp_id = SWEEP_EXPERIMENT[backbone]
    choices = load_choices(data_root)
    sweep_split = split_dir(sweep_dir(data_root) / sweep_run_name(exp_id, lr), 0)
    metrics = read_metrics(sweep_split)
    if metrics is None:
        raise ExperimentError(f"no sweep result at {sweep_split / 'metrics.json'}")
    if metrics.get("fast") or metrics.get("overrides"):
        raise ExperimentError(f"{sweep_split} is a --fast / dev-override run")
    resolved = resolve_experiment(exp_id, choices)  # the config a real run would use, with the lr now in choices.json
    expected = config_fingerprint(resolved.config, {"seed_offset": 0, "limit": None})
    if metrics.get("config_fingerprint") != expected:
        raise ExperimentError(
            f"the sweep run's config fingerprint differs from the real {resolved.config.exp_id} run "
            f"(lr in choices.json is {resolved.config.lr:g}, the sweep run used {metrics.get('lr')}, or the recipe changed)"
        )
    if metrics.get("split_sha256") != load_split(data_root, 0)["sha256"]:
        raise ExperimentError("the sweep run's split sha256 differs from the committed split_0")
    real_root = runs_dir(data_root) / run_folder_name(exp_id, choices)
    destination = split_dir(real_root, 0)
    if destination.exists() and any(destination.iterdir()):
        return "exists"
    provenance = {
        "copied_from": "/".join(sweep_split.relative_to(runs_dir(data_root)).parts),
        "reason": "choose-lr: the sweep run at the chosen lr is the same training as split 0 of this run "
        "(identical config fingerprint and split sha256), so it was copied instead of retrained",
        "config_fingerprint": expected, "split_sha256": metrics["split_sha256"], "lr": lr,
    }
    staging = destination.with_name(destination.name + f".tmp{os.getpid()}")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(sweep_split, staging)
    write_json(staging / PROVENANCE_FILENAME, provenance, indent=2)
    if destination.exists():  # empty leftover folder
        destination.rmdir()
    os.replace(staging, destination)
    return "copied"


COMPOSITE_SWEEP_EXPERIMENT = "E3-A"


def read_composite_sweep(data_root) -> tuple[dict, str]:
    """({factor: metrics of the split-0 sweep run}, split sha256). Refuses missing, --fast / dev, mismatching runs."""
    runs, shas = {}, set()
    for factor in COMPOSITE_N_FACTORS:
        run_dir = split_dir(sweep_dir(data_root) / composite_sweep_run_name(COMPOSITE_SWEEP_EXPERIMENT, factor), 0)
        metrics = read_metrics(run_dir)
        if metrics is None:
            raise ExperimentError(
                f"missing sweep result {run_dir / 'metrics.json'}; run `run --exp {COMPOSITE_SWEEP_EXPERIMENT} --sweep-composites` first")
        if metrics.get("fast") or metrics.get("overrides"):
            raise ExperimentError(f"{run_dir} is a --fast / dev-override run, not a real sweep run; re-run the sweep without those flags")
        used = (metrics.get("composites") or {}).get("n_factor")
        if used is None or abs(used - factor) > 1e-12:
            raise ExperimentError(f"{run_dir}: expected composites per epoch factor {factor:g}, found {used}")
        runs[factor] = metrics
        shas.add(metrics["split_sha256"])
    if len(shas) != 1:
        raise ExperimentError("the two composite sweep runs used different splits; re-run both")
    return runs, shas.pop()


def reuse_composite_sweep_run(data_root, factor: float) -> str:
    """Copy the sweep run at ``factor`` to split 0 of `_work/runs/E3-A/` ("copied", or "exists" when a real split 0 is
    there). ExperimentError when it is not the same training: other config fingerprint (recipe, backbone, lr, factor
    changed since) or other split."""
    data_root = Path(data_root)
    choices = load_choices(data_root)
    sweep_split = split_dir(sweep_dir(data_root) / composite_sweep_run_name(COMPOSITE_SWEEP_EXPERIMENT, factor), 0)
    metrics = read_metrics(sweep_split)
    resolved = resolve_experiment(COMPOSITE_SWEEP_EXPERIMENT, choices)  # with the factor now in choices.json
    expected = config_fingerprint(resolved.config, {"seed_offset": 0, "limit": None,
                                                    **composite_fingerprint_extra(resolved.config, resolved.composite_factor)})
    if metrics is None or metrics.get("config_fingerprint") != expected:
        raise ExperimentError(
            "the sweep run's config fingerprint differs from the real E3-A run (the recipe, backbone or lr changed since the sweep)")
    if metrics.get("split_sha256") != load_split(data_root, 0)["sha256"]:
        raise ExperimentError("the sweep run's split sha256 differs from the committed split_0")
    destination = split_dir(runs_dir(data_root) / run_folder_name(COMPOSITE_SWEEP_EXPERIMENT, choices), 0)
    if destination.exists() and any(destination.iterdir()):
        return "exists"
    provenance = {
        "copied_from": "/".join(sweep_split.relative_to(runs_dir(data_root)).parts),
        "reason": "choose-composites: the sweep run at the chosen composites-per-epoch factor is the same training as split 0 "
        "of this run (identical config fingerprint and split sha256), so it was copied instead of retrained",
        "config_fingerprint": expected, "split_sha256": metrics["split_sha256"], "n_factor": factor,
    }
    staging = destination.with_name(destination.name + f".tmp{os.getpid()}")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(sweep_split, staging)
    write_json(staging / PROVENANCE_FILENAME, provenance, indent=2)
    if destination.exists():
        destination.rmdir()
    os.replace(staging, destination)
    return "copied"


def choose_composites(data_root, out=print, *, force: bool = False) -> dict:
    """T5.3 rule: of the two split-0 sweep runs of E3-A (0.5x and 1x white-train count composites per epoch), take the
    factor with the higher **validation** mixed macro-F1; a tie goes to 1x, the default. Records it in choices.json, or
    verifies it against the recorded one when there is one (``settle_choice``; ``force`` overwrites)."""
    runs, sha = read_composite_sweep(data_root)
    scores = {f: runs[f]["val"]["mixed"]["macro_f1"] for f in COMPOSITE_N_FACTORS}
    best = max(COMPOSITE_N_FACTORS, key=lambda f: (scores[f], f == COMPOSITE_DEFAULT_FACTOR))
    evidence = {
        "experiment": COMPOSITE_SWEEP_EXPERIMENT, "split": 0, "split_sha256": sha,
        "val_mixed_macro_f1": {composite_factor_tag(f): scores[f] for f in COMPOSITE_N_FACTORS},
        "n_per_epoch": {composite_factor_tag(f): runs[f]["composites"]["n_per_epoch"] for f in COMPOSITE_N_FACTORS},
        "rule": "higher validation mixed macro-F1 on split 0; a tie goes to 1x (the default)",
    }
    out("choose-composites: " + ", ".join(f"{composite_factor_tag(f)}x -> val mixed F1 {fmt3(scores[f])}" for f in COMPOSITE_N_FACTORS)
        + f"  =>  {composite_factor_tag(best)}x")
    outcome = settle_choice(data_root, "composite_n_factor", best, evidence, force=force, command="choose-composites",
                            label="composite_n_factor", compare=lambda recorded, new: sweep_comparison(recorded, new, lambda tag: f"{tag}x"), out=out)
    if outcome in ("written", "overwritten"):
        out(f"choose-composites: composite_n_factor = {composite_factor_tag(best)} written to {choices_path(data_root)}"
            + (" (--force: it replaced the recorded one)" if outcome == "overwritten" else ""))
    chosen = load_choices(data_root).get("composite_n_factor")  # in force: the recomputed factor, or the recorded one that was kept
    try:
        reuse = reuse_composite_sweep_run(data_root, chosen)  # the sweep run at the factor in force is the same training as split 0
        message = {
            "copied": f"the sweep run at {composite_factor_tag(chosen)}x{' (the recorded factor)' if outcome == 'differs' else ''} was copied to split 0 "
            "of E3-A (same training; the real run will skip split 0)",
            "exists": "the real split 0 already exists and was left untouched",
        }[reuse]
    except (ExperimentError, DatasetError) as exc:
        reuse, message = "not reused", f"sweep run not reused for split 0 ({exc}); the real run will train it"
    out(f"choose-composites: {message}")
    return {"n_factor": chosen, "recomputed_n_factor": best, "evidence": evidence, "split_0": reuse, "outcome": outcome}


def cmd_choose_composites(args: argparse.Namespace) -> int:
    try:
        choose_composites(Path(args.data_root), force=args.force)
    except (ExperimentError, DatasetError) as exc:
        print(f"choose-composites: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK


def cmd_choose_lr(args: argparse.Namespace) -> int:
    try:
        choose_lr(Path(args.data_root), args.backbone, force=args.force)
    except (ExperimentError, DatasetError) as exc:
        print(f"choose-lr: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

In [ ]:
#| export
def cmd_predict(args: argparse.Namespace) -> int:
    data_root = Path(args.data_root)
    try:
        if args.run_dir is not None:
            run_dir = Path(args.run_dir)
        else:
            choices = load_choices(data_root)
            exp_id = args.exp or choices.raw("recommended")
            if not exp_id:
                raise ExperimentError(
                    f"no --exp given and {choices.path} has no 'recommended' experiment (written by gate G2, phase 06). "
                    "Pass --exp ID, for example --exp E1-U, or --run-dir PATH"
                )
            run_dir = split_dir(runs_dir(data_root) / run_folder_name(exp_id, choices), args.split)
        if not Path(args.image).is_file():
            raise ExperimentError(f"image {args.image} does not exist")
        predict_image(run_dir, args.image, resolve_device(args.device), data_root, seg_dir=args.seg_dir)
    except (ExperimentError, DatasetError) as exc:
        print(f"predict: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK


def cmd_gradcam(args: argparse.Namespace) -> int:
    """T7.5: Grad-CAM figures and the shortcut check on the annotated field images (see `run_gradcam`)."""
    try:
        run_gradcam(Path(args.data_root), args.exp, args.compare, resolve_device(args.device), args.split, args.runs_dir)
    except (ExperimentError, DatasetError, RuntimeError) as exc:
        print(f"gradcam: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

## `gate` and `summarize` (phase 04 / 06 / 07)

`gate g0` and `gate g1` recompute a decision of phase 04 from the run folders and write it to `choices.json`
(`input_mode`, resp. `backbone`) together with the evidence; `gate g2` (phase 06) writes `recommended`, freezes it
(`--force` to overwrite a different earlier recommendation) and accepts `--cut A` / `--cut B` for a method dropped by a
cut line; see `50_analysis.ipynb`. `summarize` regenerates `_work/results/` (tables and `findings.md`) from the run
folders, skipping incomplete experiments with a warning.

`benchmark` times the forward pass of the three backbones on the device given by `--device` (default: the best one here) and merges
the result into `_work/results/inference_benchmark.json`, which `summarize` reads for `table_runtime` (it never times anything itself).

In [ ]:
#| export
def cmd_gate(args: argparse.Namespace) -> int:
    try:
        run_gate(Path(args.data_root), args.gate, cut=tuple(args.cut or ()), force=args.force)
    except (ExperimentError, DatasetError) as exc:
        print(f"gate {args.gate}: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK


def cmd_summarize(args: argparse.Namespace) -> int:
    try:
        summarize(Path(args.data_root))
    except (ExperimentError, DatasetError) as exc:
        print(f"summarize: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK


def cmd_benchmark(args: argparse.Namespace) -> int:
    try:
        run_benchmark(Path(args.data_root), resolve_device(args.device))
    except (ExperimentError, DatasetError, RuntimeError) as exc:
        print(f"benchmark: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

In [ ]:
#| export
def resolve_device(requested: str = "auto") -> str:
    """Map the --device flag to a concrete torch device name; raise if it is unavailable."""
    import torch

    available = {
        "cuda": torch.cuda.is_available(),
        "mps": torch.backends.mps.is_available(),
        "cpu": True,
    }
    if requested == "auto":
        return next(name for name in ("cuda", "mps", "cpu") if available[name])
    if not available.get(requested, False):
        raise RuntimeError(f"device {requested!r} was requested but is not available here")
    return requested

In [ ]:
#| export
def run_prepare(data_root, workers: int | None = None, out=print) -> None:
    """Index, cache, near-duplicate audit, splits, tables and figures (phase 02), `choices.json` from the frozen decisions when it
    does not exist yet, then the white pseudo-masks and QC sheets (T5.1), the composite figures (T5.2) and the annotation selection
    (T6.1). Idempotent."""
    data_root = Path(data_root)
    prepare_data(data_root, workers=workers)
    seed_choices(data_root, out=out)
    records = load_index(data_root)
    prepare_pseudo_masks(data_root, records, workers=workers)
    prepare_composites(data_root, records)
    prepare_annotations(data_root, records)


def cmd_prepare(args: argparse.Namespace) -> int:
    """`run_prepare` with a timer and an error message. Idempotent."""
    started = time.time()
    try:
        run_prepare(Path(args.data_root), args.workers)
    except DatasetError as exc:
        print(f"prepare: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    print(f"prepare: done in {time.time() - started:.0f} s")  # the last line: every step above has finished
    return EXIT_OK


def smoke_split(selected: list[Record], seed: int = 0) -> dict:
    """A deterministic tiny split of the smoke images: every class x domain stratum contributes to train, and
    val / test get the other members alternately (the real splits need >= 7 images per stratum, the smoke
    images have ~2)."""
    parts: dict[str, list[str]] = {"train": [], "val": [], "test": []}
    by_stratum: dict = {}
    for record in sorted(selected, key=lambda r: r.filename):
        by_stratum.setdefault(record.stratum, []).append(record.filename)
    for (cls, domain), names in by_stratum.items():
        even = (CLASS_NAMES.index(cls) + DOMAINS.index(domain)) % 2 == 0
        cycle = ("train", "val" if even else "test", "test" if even else "val", "train")
        for k, name in enumerate(names):
            parts[cycle[k % 4]].append(name)
    train, val, test = (sorted(parts[key]) for key in ("train", "val", "test"))
    return {"seed": seed, "train": train, "val": val, "test": test, "sha256": split_sha256(seed, train, val, test)}


def classifier_smoke(data_root, tmp_root, selected: list[Record], log=print) -> None:
    """One tiny experiment (E0-rect, ResNet50, 1 epoch per stage, input size / 4) through the real `run` code
    path on CPU, with a complete run folder written under ``tmp_root``; then `predict` on one image of it."""
    started = time.time()
    write_index(tmp_root, selected)
    split = smoke_split(selected)
    smoke_require(all(split[k] for k in ("train", "val", "test")), "tiny split has an empty part")
    write_json(split_path(tmp_root, 0), split)
    resolved = resolve_experiment("E0-rect", Choices(choices_path(tmp_root), {}), lr=3e-4)
    pretrained = pretrained_available(resolved.config.backbone, data_root)
    resolved = replace(resolved, config=replace(resolved.config, pretrained=pretrained, **FAST_CONFIG))
    options = RunOptions(device="cpu", num_workers=2, weights_root=Path(data_root))
    run_dir = split_dir(tmp_root / WORK_DIRNAME / "runs" / "E0-rect_smoke", 0)
    lines: list[str] = []
    result = run_split(resolved, 0, data_root=tmp_root, out_dir=run_dir, options=options, log=lines.append)
    smoke_require(result["status"] == "done", "the tiny run did not train")
    for name in (*RUN_FILES, "best.pt"):
        smoke_require((run_dir / name).is_file(), f"run folder lacks {name}")
    smoke_require(not list(run_dir.glob("*.tmp*")), "temporary files left behind (non-atomic write)")
    metrics = read_metrics(run_dir)
    smoke_require(metrics["split_sha256"] == split["sha256"] and metrics["wall_time_s"] > 0, "metrics.json is incomplete")
    for part in ("val", "test"):
        with open(run_dir / f"predictions_{part}.csv", newline="") as handle:
            rows = list(csv.reader(handle))
        smoke_require(rows[0][:4] == ["filename", "domain", "true", "pred"] and len(rows) - 1 == len(split[part]),
                      f"predictions_{part}.csv has the wrong rows")
    again = run_split(resolved, 0, data_root=tmp_root, out_dir=run_dir, options=options, log=lines.append)
    smoke_require(again["status"] == "skipped", "a finished split was not skipped (idempotence)")
    image = Path(data_root) / selected[0].rel_path
    probs = predict_image(run_dir, image, "cpu", data_root, out=lambda text: None)
    smoke_require(probs.shape == (NUM_CLASSES,) and abs(float(probs.sum()) - 1.0) < 1e-4, "predict returned bad probabilities")
    log(f"smoke: OK - tiny {resolved.config.exp_id} run ({'pretrained' if pretrained else 'random'} weights, "
        f"{len(split['train'])}/{len(split['val'])}/{len(split['test'])} train/val/test images, workers 2): complete run folder, "
        f"skipped on rerun, predict OK ({time.time() - started:.0f} s)")


def method_a_smoke(data_root, tmp_root, selected: list[Record], log=print) -> None:
    """Method A on the tiny smoke set (everything in ``tmp_root``): pseudo-masks of the white images and a QC sheet,
    one composite per class (reproducible from its seed), and one tiny E3-A run whose training set contains
    composites, through the real `run_split` path."""
    started = time.time()
    stats = build_white_masks(tmp_root, selected, workers=1)
    white = [r for r in selected if r.domain == "white"]
    smoke_require(stats.total == len(white) and stats.built == len(white), "pseudo-masks were not built for every white image")
    for record in white:
        mask = read_white_mask(tmp_root, record.filename)
        smoke_require(mask.shape == read_cached_rgb(tmp_root, record.filename).shape[:2] and mask.any(),
                      f"{record.filename}: the pseudo-mask is empty or has the wrong size")
    smoke_require(build_white_masks(tmp_root, selected, workers=1).built == 0, "second mask run is not idempotent")
    sheets = save_mask_qc_sheets(tmp_root, selected, rejects=set())
    smoke_require(all(p.stat().st_size > 1000 for p in sheets), "QC sheets were not written")
    split = smoke_split(selected)
    pool = CompositePool(tmp_root, 0, selected, exclusions=set())
    smoke_require(pool.leaves and all(r.filename in split["train"] for r in pool.leaves), "composite pool is empty or leaks")
    classes = sorted({r.cls_idx for r in pool.leaves})
    for cls_idx in classes:
        leaf, background = pool.sample(np.random.default_rng(cls_idx), cls_idx)
        image, mask, label = make_composite(leaf, background, np.random.default_rng(7), data_root=tmp_root)
        again, _, _ = make_composite(leaf, background, np.random.default_rng(7), data_root=tmp_root)
        smoke_require(label == cls_idx and image.shape[:2] == mask.shape and mask.any() and np.array_equal(image, again),
                      f"composite of class {cls_idx} is empty or not reproducible")
    resolved = resolve_experiment("E3-A", Choices(choices_path(tmp_root), {}), placeholders=True)
    resolved = replace(resolved, config=replace(resolved.config, pretrained=False, **FAST_CONFIG))
    run_dir = split_dir(tmp_root / WORK_DIRNAME / "runs" / "E3-A_smoke", 0)
    result = run_split(resolved, 0, data_root=tmp_root, out_dir=run_dir, options=RunOptions(device="cpu", num_workers=0), log=lambda _: None)
    metrics = result["metrics"]
    smoke_require(metrics["composites"]["n_per_epoch"] > 0 and metrics["n_train"] > len(split["train"]) - 1, "E3-A trained without composites")
    log(f"smoke: OK - Method A: {stats.built} pseudo-masks and QC sheet, composites of {len(classes)} classes (reproducible), "
        f"tiny E3-A run with {metrics['composites']['n_per_epoch']} composites per epoch ({time.time() - started:.0f} s)")


def method_b_smoke(data_root, tmp_root, selected: list[Record], log=print) -> None:
    """Method B on the tiny smoke set (everything in ``tmp_root``, after `method_a_smoke` built the pseudo-masks): one tiny
    segmenter epoch through `run_seg_split` (run folder, boxes of every image, no annotation touched), a tiny E4-B run that
    crops by those boxes, and `predict` on the E4-B run (segmenter, crop, classifier)."""
    started = time.time()
    resolved = resolve_experiment("S-seg", Choices(choices_path(tmp_root), {}))
    pretrained = pretrained_available(resolved.config.backbone, data_root)
    resolved = replace(resolved, config=replace(resolved.config, pretrained=pretrained, **FAST_CONFIG))
    seg_run = split_dir(runs_dir(tmp_root) / "S-seg", 0)
    result = run_seg_split(resolved, 0, data_root=tmp_root, out_dir=seg_run,
                           options=RunOptions(device="cpu", num_workers=0, limit=FAST_IMAGES, weights_root=Path(data_root)), log=lambda _: None)
    for name in ("config.json", "env.json", "history.csv", "metrics.json", "best.pt", BOXES_FILENAME, SEG_METRICS_FILENAME):
        smoke_require((seg_run / name).is_file(), f"segmenter run folder lacks {name}")
    smoke_require(not list(seg_run.glob("*.tmp*")), "temporary files left behind (segmenter run)")
    seg_metrics = read_seg_metrics(seg_run)
    smoke_require("dev" not in seg_metrics and "eval" not in seg_metrics, "a training run must not touch the annotations")
    boxes = load_boxes(tmp_root, 0)
    smoke_require(set(boxes) == {r.filename for r in selected}, "boxes.json does not cover every cached image")
    smoke_require(all(e["fallback"] or (e["box"][0] < e["box"][2] and e["box"][1] < e["box"][3]) for e in boxes.values()), "a box has no area")
    smoke_require(run_seg_split(resolved, 0, data_root=tmp_root, out_dir=seg_run, options=RunOptions(device="cpu", num_workers=0, limit=FAST_IMAGES,
                  weights_root=Path(data_root)), log=lambda _: None)["status"] == "skipped", "a finished segmenter split was not skipped")
    b = resolve_experiment("E4-B", Choices(choices_path(tmp_root), {}), placeholders=True)
    b = replace(b, config=replace(b.config, pretrained=False, **FAST_CONFIG))
    run_dir = split_dir(runs_dir(tmp_root) / "E4-B_smoke", 0)
    outcome = run_split(b, 0, data_root=tmp_root, out_dir=run_dir, options=RunOptions(device="cpu", num_workers=0), log=lambda _: None)
    smoke_require(outcome["metrics"]["seg_boxes_sha256"] == result["metrics"]["boxes_sha256"], "E4-B did not record the boxes it used")
    probs = predict_image(run_dir, Path(data_root) / selected[0].rel_path, "cpu", tmp_root, out=lambda text: None)
    smoke_require(probs.shape == (NUM_CLASSES,) and abs(float(probs.sum()) - 1.0) < 1e-4, "predict (E4-B) returned bad probabilities")
    fallbacks = result["metrics"]["fallbacks"]
    log(f"smoke: OK - Method B: tiny segmenter epoch ({'pretrained' if pretrained else 'random'} weights, {result['metrics']['n_train_per_epoch']} samples), "
        f"boxes for {len(boxes)} images ({sum(c['fallback'] for c in fallbacks.values())} fallbacks), tiny E4-B run cropping by them, "
        f"predict = segment + crop + classify ({time.time() - started:.0f} s)")


def reproduce_smoke(data_root, tmp_root, log=print) -> None:
    """`reproduce --recommended --fast --splits 0` on the tiny smoke set (everything in ``tmp_root``, after `classifier_smoke` wrote its
    index and split): `choices.json` is seeded from the frozen decisions, then the command's own steps run in-process - a tiny run of the
    recommended model into scratch, and `summarize`. (`download-weights` and `prepare` need the real dataset; the tests cover them.)"""
    started = time.time()
    smoke_require(seed_choices(tmp_root, out=lambda text: None), "choices.json was not seeded from the frozen decisions")
    smoke_require(json.loads(read_file_bytes(choices_path(tmp_root)).decode("utf-8")) == FROZEN_CHOICES, "the seeded choices.json differs from FROZEN_CHOICES")
    smoke_require(not seed_choices(tmp_root, out=lambda text: None), "an existing choices.json was seeded again")
    args = build_parser().parse_args(["reproduce", "--recommended", "--fast", "--splits", "0", "--data-root", str(tmp_root), "--device", "cpu"])
    steps = [step for step in reproduce_steps(args) if step.argv[0] in ("run", "summarize")]
    lines: list[str] = []
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        code = run_reproduce(args, steps, out=lines.append)
    smoke_require(code == EXIT_OK and any("2 of 2 steps done" in line for line in lines), f"reproduce --recommended --fast failed: {lines[-3:]}")
    recommended = FROZEN_CHOICES["recommended"]["value"]
    metrics = read_metrics(split_dir(scratch_dir(tmp_root) / f"{recommended}_fast", 0))
    smoke_require(metrics is not None and metrics["fast"], f"reproduce trained no tiny {recommended} run")
    smoke_require((results_dir(tmp_root) / "table_required.md").is_file(), "reproduce wrote no tables (summarize)")
    log(f"smoke: OK - reproduce --recommended --fast: choices.json seeded from the frozen decisions, tiny {recommended} run, summarize ({time.time() - started:.0f} s)")


def smoke_extra_checks(data_root, tmp_root, selected: list[Record]) -> None:
    classifier_smoke(data_root, tmp_root, selected)
    method_a_smoke(data_root, tmp_root, selected)
    method_b_smoke(data_root, tmp_root, selected)
    reproduce_smoke(data_root, tmp_root)


def cmd_smoke(args: argparse.Namespace) -> int:
    """Tiny end-to-end check: data root, image discovery, library imports, device, the data path (index, EXIF
    read, cache, dHash, splits, figures on ~20 images in a temp folder) and one tiny classifier experiment
    through the real `run` code path on CPU, followed by `predict`, then Method A (pseudo-masks, composites, a tiny E3-A run)
    and Method B (a tiny segmenter epoch, boxes, a tiny E4-B run, `predict` with the segmenter); everything in a temp folder."""
    data_root = Path(args.data_root)
    print(f"smoke: data root = {data_root.resolve()}")
    if not data_root.is_dir():
        print(f"smoke: FAIL - data root {data_root} is not a directory (see --data-root)", file=sys.stderr)
        return EXIT_FAILURE
    images = find_images(data_root)
    if not images:
        print(f"smoke: FAIL - no JPEG images found under {data_root}", file=sys.stderr)
        return EXIT_FAILURE
    print(f"smoke: OK - found {len(images)} JPEG images (excluding {WORK_DIRNAME}/)")

    import cv2
    import matplotlib
    import numpy
    import skimage
    import sklearn
    import torch
    import torchvision

    print(f"smoke: python {sys.version.split()[0]}")
    for module in (numpy, matplotlib, cv2, sklearn, skimage, torch, torchvision):
        print(f"smoke: {module.__name__} {module.__version__}")
    device = resolve_device(args.device)
    print(f"smoke: OK - device = {device} (requested: {args.device}), fast = {args.fast}")
    try:
        run_data_smoke(data_root, extra_checks=lambda tmp_root, selected: smoke_extra_checks(data_root, tmp_root, selected))
    except (DatasetError, ExperimentError) as exc:
        print(f"smoke: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    print("smoke: PASSED")
    return EXIT_OK

## `reproduce` (phase 08)

`reproduce` is the one command the user manual gives the marker. It runs, **in-process and in this order**, the functions behind the
individual commands, so a step does exactly what the same command line does:

| # | Step | Why here |
|---|---|---|
| 1-2 | `download-weights`, `prepare` | weights; index, cache, splits, pseudo-masks, annotations; seeds `choices.json` from `FROZEN_CHOICES` |
| 3-4 | `run --exp E0-rect --sweep-lr`, `choose-lr --backbone resnet50` | learning rate of ResNet50 (split 0, 1e-4 vs 3e-4); copies the winner to split 0 of E0-rect |
| 5-6 | `run --exp E0`, `gate g0` | input-size ablation (squash, pad, rect), then gate G0 |
| 7 | `run --exp E1` | E1-U (alias of the winning E0 row), E1-WW (alias of E1-WF), E1-FF, E1-WF |
| 8-11 | `run --exp E2-cnx-U --sweep-lr`, `run --exp E2-swin-U --sweep-lr`, `choose-lr` for `convnext_tiny` and `swin_t` | learning rates of the two other backbones |
| 12-13 | `run --exp E2`, `gate g1` | ConvNeXt-T and Swin-T (U and W->F rows), then gate G1 |
| 14-16 | `run --exp E3-A --sweep-composites`, `choose-composites`, `run --exp E3-A,E3-A-WF` | Method A |
| 17 | `gate g2 --cut B` | recommended method (Method B was cut, see the findings) |
| 18-19 | `benchmark`, `gradcam` | inference time per image (`table_runtime`); Grad-CAM tables and figures (needs the `best.pt` of E2-swin-U and E3-A that the runs above keep) |
| 20 | `summarize` | every table, figure and `findings.md` |

Not in the chain, on purpose: Method B (`S-seg`, `E4-B`; cut at gate G2), `seg-eval` and the X-domain rows (not reported).

`reproduce --recommended` is the quick check: `download-weights`, `prepare`, `run --exp <recommended>` (the model gate G2 froze,
read from `choices.json`), `summarize`. The gates and choosers of the chain **verify** the frozen decisions (see the section on
`choices.json`): they never change what the later steps train, so the reproduced tables are comparable with the reported ones.
`gate g2` runs non-strict here (`strict=False`): another outcome keeps the frozen recommendation and warns instead of stopping.

`--splits` goes to every `run` step except the sweeps (always split 0). With fewer than all five splits the gates and Grad-CAM,
which need complete experiments, are skipped with a message (the frozen decisions stay in force). `--fast` is only for
`--recommended` (a fast run is never a result, so the full chain would have nothing to read) and skips `download-weights`.

Resuming = running the same command again: `run` skips finished splits, the choosers and gates verify. A failing step stops the
chain with its number, and the message says so.

In [ ]:
#| export
@dataclass(frozen=True)
class Step:
    """One step of `reproduce`: the command line of an individual command, without the global flags."""

    argv: tuple[str, ...]
    needs_all_splits: bool = False  # reads complete 5-split experiments (gates, Grad-CAM): skipped when --splits asks for fewer

    @property
    def label(self) -> str:
        return " ".join(self.argv)


# The reported chain, in the order the decisions depend on each other (see the section text).
REPRODUCE_CHAIN = (
    Step(("download-weights",)),
    Step(("prepare",)),
    Step(("run", "--exp", "E0-rect", "--sweep-lr")),
    Step(("choose-lr", "--backbone", "resnet50")),
    Step(("run", "--exp", "E0")),
    Step(("gate", "g0"), needs_all_splits=True),
    Step(("run", "--exp", "E1")),
    Step(("run", "--exp", "E2-cnx-U", "--sweep-lr")),
    Step(("run", "--exp", "E2-swin-U", "--sweep-lr")),
    Step(("choose-lr", "--backbone", "convnext_tiny")),
    Step(("choose-lr", "--backbone", "swin_t")),
    Step(("run", "--exp", "E2")),
    Step(("gate", "g1"), needs_all_splits=True),
    Step(("run", "--exp", "E3-A", "--sweep-composites")),
    Step(("choose-composites",)),
    Step(("run", "--exp", "E3-A,E3-A-WF")),
    Step(("gate", "g2", "--cut", "B"), needs_all_splits=True),
    Step(("benchmark",)),
    Step(("gradcam",), needs_all_splits=True),
    Step(("summarize",)),
)
# Measured on one T4 (table_runtime): E2-swin-U takes 7.6 min per split; the sum over the chain is about 9 h plus the one-off prepare.
REPRODUCE_DESCRIPTION = (
    "Reproduce the reported results from the original dataset (Dhan-Shomadhan/ next to this file), one step after the other,\n"
    "in this process. Every step is also a command of its own.\n"
    "\n"
    "modes:\n"
    "  reproduce                the whole chain below: every reported experiment on 5 splits, then all tables and figures\n"
    "                           (about 10 hours on one T4 GPU)\n"
    f"  reproduce --recommended  the quick check: download-weights, prepare, run --exp {FROZEN_CHOICES['recommended']['value']} (the recommended model),\n"
    "                           summarize (about 40 minutes on one T4 GPU, plus the one-off prepare)\n"
    "  --splits LIST            any mode: only these splits of every run (--splits 0: about a fifth of the training; the gates and\n"
    "                           Grad-CAM need all 5 splits and are skipped)\n"
    "\n"
    "The decisions (input mode, learning rates, backbone, composites, recommended model) are frozen in this file. The gates and\n"
    "choosers recompute them from your runs and check them against the frozen ones; a difference is printed as a warning and the\n"
    "frozen decision is kept, so that the tables you get are the reported ones.\n"
    "\n"
    "It can be stopped and started again with the same command: finished splits are skipped. A failing step stops the chain and\n"
    "says which one.\n"
    "\n"
    "the chain (reproduce):\n"
    + "\n".join(f"  {number:>2}. {step.label}" for number, step in enumerate(REPRODUCE_CHAIN, 1))
)


def recommended_experiment_id(data_root) -> str:
    """The model of `reproduce --recommended`: the one in choices.json, else the frozen one that `prepare` is going to seed."""
    choices = load_choices(data_root)
    return choices.get("recommended") if choices.raw("recommended") is not None else FROZEN_CHOICES["recommended"]["value"]


def reproduce_steps(args) -> list[Step]:
    """The steps of this invocation: the whole chain, or with ``--recommended`` weights, prepare, the recommended run, summarize."""
    if not args.recommended:
        return list(REPRODUCE_CHAIN)
    return [Step(("download-weights",)), Step(("prepare",)), Step(("run", "--exp", recommended_experiment_id(Path(args.data_root)))),
            Step(("summarize",))]


def step_namespace(step: Step, args) -> argparse.Namespace:
    """The arguments the command of ``step`` would get on the command line: its own flags, ``--splits`` for the real runs (the
    sweeps are split 0 anyway) and the global flags of `reproduce`."""
    argv = list(step.argv)
    if argv[0] == "run" and "--sweep-lr" not in argv and "--sweep-composites" not in argv:
        argv += ["--splits", ",".join(str(k) for k in args.splits)]
    argv += ["--data-root", str(args.data_root), "--device", args.device, "--seed-offset", str(args.seed_offset)]
    if args.fast:
        argv.append("--fast")
    return build_parser().parse_args(argv)


def execute_step(step_args: argparse.Namespace) -> str | None:
    """Run the command of one step in-process, through the same functions as its own command. Returns "written", "matches",
    "differs" or "overwritten" for the gates and choosers, else None. Errors propagate. The gates are not strict: a different
    outcome keeps the frozen decision and warns, so that the chain goes on with the reported configuration."""
    data_root, command = Path(step_args.data_root), step_args.command
    if command == "download-weights":
        download_weights(data_root)
    elif command == "prepare":
        run_prepare(data_root, step_args.workers)
    elif command == "run":
        run_command(step_args)
    elif command == "choose-lr":
        return choose_lr(data_root, step_args.backbone, force=step_args.force)["outcome"]
    elif command == "choose-composites":
        return choose_composites(data_root, force=step_args.force)["outcome"]
    elif command == "gate":
        return run_gate(data_root, step_args.gate, cut=tuple(step_args.cut or ()), force=step_args.force, strict=False)["outcome"]
    elif command == "benchmark":
        run_benchmark(data_root, resolve_device(step_args.device))
    elif command == "gradcam":
        run_gradcam(data_root, step_args.exp, step_args.compare, resolve_device(step_args.device), step_args.split, step_args.runs_dir)
    elif command == "summarize":
        summarize(data_root)
    else:
        raise ExperimentError(f"reproduce has no step for {command!r}")
    return None


def skip_reason(step: Step, args) -> str | None:
    """Why a step is not run in this invocation (None: it is)."""
    if step.argv[0] == "download-weights" and args.fast:
        return "--fast uses the pretrained weights only if they are already downloaded"
    if step.needs_all_splits and set(args.splits) != set(DEFAULT_SPLITS):
        return (f"it needs all five splits (--splits {','.join(str(k) for k in args.splits)}); "
                "the frozen decision stays in force")
    return None


def format_elapsed(seconds: float) -> str:
    """'12 s', '5 min 03 s' or '2 h 05 min'."""
    hours, rest = divmod(int(round(seconds)), 3600)
    minutes, secs = divmod(rest, 60)
    if hours:
        return f"{hours} h {minutes:02d} min"
    return f"{minutes} min {secs:02d} s" if minutes else f"{secs} s"


def run_reproduce(args, steps: list[Step] | None = None, out=print) -> int:
    """`reproduce` (see the section text). ``steps`` replaces the step list (tests). Returns the exit code: the first failing step
    stops the chain."""
    data_root = Path(args.data_root)
    if args.fast and not args.recommended:
        print("reproduce: FAIL - --fast only works with --recommended: a fast run is never a result, so the sweeps and gates of the "
              "whole chain would have nothing to read", file=sys.stderr)
        return EXIT_FAILURE
    try:
        steps = reproduce_steps(args) if steps is None else steps
    except ExperimentError as exc:
        print(f"reproduce: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    total = len(steps)
    started = time.time()
    out(f"reproduce: {'recommended model only' if args.recommended else 'the whole reported chain'}: {total} steps, "
        f"splits {','.join(str(k) for k in args.splits)}, data root {data_root}")
    done: list[tuple[str, str | None]] = []  # (header, outcome) of the steps that ran
    skipped = 0
    for number, step in enumerate(steps, 1):
        header = f"[{number}/{total}] {step.label}"
        out(f"\nreproduce: {header}")
        reason = skip_reason(step, args)
        if reason is not None:
            skipped += 1
            out(f"reproduce: {header} skipped - {reason}")
            continue
        began = time.time()
        try:
            outcome = execute_step(step_namespace(step, args))
        except KeyboardInterrupt:
            print(f"\nreproduce: interrupted at step {header} after {format_elapsed(time.time() - started)}. "
                  "Run the same command again to resume: finished splits are skipped.", file=sys.stderr)
            return 130
        except (ExperimentError, DatasetError, RuntimeError, OSError) as exc:
            print(f"reproduce: FAIL at step {header} after {format_elapsed(time.time() - started)} - {exc}", file=sys.stderr)
            print(f"reproduce: the {len(done)} step(s) before it are done. Fix the problem and run the same command again: finished splits "
                  "are skipped and the decisions are checked again, so it resumes at this step.", file=sys.stderr)
            return EXIT_FAILURE
        done.append((header, outcome))
        out(f"reproduce: {header} done in {format_elapsed(time.time() - began)} (total {format_elapsed(time.time() - started)})")
    out(f"\nreproduce: summary - {len(done)} of {total} steps done" + (f", {skipped} skipped" if skipped else "")
        + f" in {format_elapsed(time.time() - started)}")
    checks = [(header, outcome) for header, outcome in done if outcome is not None]
    if checks:
        out("reproduce: decisions recomputed from your runs, against the frozen ones (the frozen ones stay in force):")
        words = {"matches": "matches", "differs": "DIFFERS (kept the frozen one, see the warning above)",
                 "written": "no frozen decision, recorded", "overwritten": "overwritten (--force)"}
        for header, outcome in checks:
            out(f"  {header}: {words[outcome]}")
        different = sum(outcome == "differs" for _, outcome in checks)
        if different:
            out(f"reproduce: {different} decision(s) differ from the frozen ones (numbers in the warnings above); "
                "the later experiments used the frozen decisions.")
    out(f"reproduce: tables and findings: {results_dir(data_root)}; figures: {figures_dir(data_root)}")
    return EXIT_OK


def cmd_reproduce(args: argparse.Namespace) -> int:
    return run_reproduce(args)

In [ ]:
#| export
def parse_splits(text: str) -> tuple[int, ...]:
    """Parse '0,1,2' into (0, 1, 2); duplicates are dropped, order is kept."""
    try:
        values = [int(part) for part in text.split(",") if part.strip() != ""]
    except ValueError:
        raise argparse.ArgumentTypeError(f"expected comma-separated integers, got {text!r}") from None
    if not values or min(values) < 0:
        raise argparse.ArgumentTypeError(f"expected one or more non-negative integers, got {text!r}")
    return tuple(dict.fromkeys(values))


def add_global_flags(parser: argparse.ArgumentParser, *, is_subcommand: bool) -> None:
    """Add the global flags. On subparsers the defaults are suppressed so that a flag given
    before the command is not overwritten by the subparser's default."""

    def default(value):
        return argparse.SUPPRESS if is_subcommand else value

    group = parser.add_argument_group("global flags (accepted before or after the command)")
    group.add_argument(
        "--data-root",
        type=Path,
        default=default(Path(DEFAULT_DATA_ROOT)),
        help=f"dataset folder, relative to the working directory (default: {DEFAULT_DATA_ROOT})",
    )
    group.add_argument(
        "--device",
        choices=DEVICE_CHOICES,
        default=default("auto"),
        help="compute device (default: auto = cuda, then mps, then cpu)",
    )
    group.add_argument(
        "--seed-offset",
        type=int,
        default=default(0),
        help="added to every seed, for repeating an experiment with different randomness (default: 0)",
    )
    group.add_argument(
        "--fast",
        action="store_true",
        default=default(False),
        help="smoke settings: tiny subsets, 1 epoch",
    )


def _configure_prepare(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--workers",
        type=int,
        default=None,
        metavar="N",
        help="worker processes for image caching (default: number of CPU cores, at most 8)",
    )


def _configure_run(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--exp", required=True, metavar="ID", help="experiment ID (E1-U), a group (E0, E1, E2, X-domain) or a comma-separated list")
    parser.add_argument(
        "--splits",
        type=parse_splits,
        default=DEFAULT_SPLITS,
        metavar="LIST",
        help="comma-separated split seeds (default: 0,1,2,3,4)",
    )
    parser.add_argument("--force", action="store_true", help="retrain splits whose metrics.json already exists")
    parser.add_argument("--sweep-lr", action="store_true",
                        help="learning-rate sweep: split 0 at lr 1e-4 and 3e-4 into _work/runs/sweep/ (never a result)")
    parser.add_argument("--sweep-composites", action="store_true",
                        help="Method A: split 0 of --exp E3-A with 0.5x and 1x composites per epoch into _work/runs/sweep/ (never a result)")
    parser.add_argument("--batch-size", type=int, default=None, metavar="N", help="batch size (default 32; 16 + --grad-accum 2 if memory is short)")
    parser.add_argument("--grad-accum", type=int, default=None, metavar="N", help="gradient accumulation steps (default 1)")
    parser.add_argument("--num-workers", type=int, default=None, metavar="N", help="data-loading processes (default: cores - 1, at most 4)")
    parser.add_argument("--out-dir", type=Path, default=None, metavar="DIR", help="write DIR/<ID>/split_<k>/ instead of _work/runs/<ID>/")
    dev = parser.add_argument_group("dev overrides (output goes to _work/runs/scratch/, never counted as a result)")
    dev.add_argument("--lr", type=float, default=None, metavar="LR", help="fine-tuning learning rate instead of choices.json")
    dev.add_argument("--epochs", type=int, default=None, metavar="N", help="fine-tuning epochs (default 30)")
    dev.add_argument("--stage1-epochs", type=int, default=None, metavar="N", help="linear-probe epochs (default 2)")
    dev.add_argument("--seg-arch", default=None, choices=tuple(SEG_ARCHITECTURES), metavar="ARCH",
                     help="S-seg only: try another segmenter architecture (the one swap 06 allows, decided on the dev annotations)")
    parser.add_argument("--seg-dir", type=Path, default=None, metavar="DIR",
                        help="Method B (E4-B): the S-seg runs folder whose split_<k>/boxes.json to use (default: _work/runs/S-seg)")


def _configure_reproduce(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--recommended", action="store_true",
                        help="only the recommended model: download-weights, prepare, run --exp <the model of choices.json>, summarize "
                             "(about 40 minutes on one T4 GPU, plus the one-off prepare)")
    parser.add_argument("--splits", type=parse_splits, default=DEFAULT_SPLITS, metavar="LIST",
                        help="comma-separated split seeds of every run (default: 0,1,2,3,4; the sweeps are split 0 anyway)")


def _configure_choose_lr(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--backbone", required=True, choices=BACKBONES, help="backbone whose split-0 sweep to evaluate")
    parser.add_argument("--force", action="store_true",
                        help="overwrite a recorded learning rate that differs from the recomputed one (default: keep the recorded one and warn)")


def _configure_choose_composites(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--force", action="store_true",
                        help="overwrite a recorded composites factor that differs from the recomputed one (default: keep the recorded one and warn)")


def _configure_gate(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("gate", choices=("g0", "g1", "g2"), help="g0: input mode (after E0), g1: backbone (after E1-U, E2-cnx-U, E2-swin-U), "
                        "g2: recommended method (plain G1 model vs E3-A vs E4-B)")
    parser.add_argument("--cut", action="append", choices=("A", "B"), default=None, metavar="METHOD",
                        help="g2 only: leave Method A or B out because it was cut (cut lines of the README); repeat for both")
    parser.add_argument("--force", action="store_true",
                        help="overwrite a recorded decision that differs from the recomputed one (default: g0 and g1 keep the recorded one and warn, "
                             "g2 refuses)")


def _configure_gradcam(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--exp", default=None, metavar="ID",
                        help="the plain model, whose T7.4 success and failure images are shown (default: the recommended one in choices.json)")
    parser.add_argument("--compare", default=GRADCAM_COMPARE_EXPERIMENT, metavar="ID",
                        help=f"the model compared with it (default: {GRADCAM_COMPARE_EXPERIMENT}, Method A)")
    parser.add_argument("--split", type=int, default=None, metavar="K",
                        help="use split K only and add _splitK to every file name (default: every image on a split where it was a test image, "
                             "annotated images on every split where they were test or validation; findings.md is only written without --split)")
    parser.add_argument("--runs-dir", type=Path, default=None, metavar="DIR",
                        help="DIR/<run folder>/split_<k>/best.pt instead of the retrained checkpoints (default: _work/runs/<run folder>/ if it kept "
                             "best.pt, else _work/runs/retrain/<run folder>/)")


def _configure_predict(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--exp", default=None, metavar="ID", help="experiment ID (default: the recommended one in choices.json)")
    parser.add_argument("--split", type=int, default=0, metavar="K", help="split seed (default: 0)")
    parser.add_argument("--image", type=Path, required=True, metavar="PATH", help="image to classify")
    parser.add_argument("--run-dir", type=Path, default=None, metavar="DIR", help="use DIR/best.pt directly instead of the run folder of --exp")
    parser.add_argument("--seg-dir", type=Path, default=None, metavar="DIR",
                        help="Method B: the S-seg runs folder with the segmenter checkpoints (default: next to the experiment's folder)")


def _configure_seg_eval(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--set", required=True, choices=SEG_EVAL_SETS,
                        help="dev: the 10 development annotations (any time); eval: the 20 reporting annotations (needs --final)")
    parser.add_argument("--splits", type=parse_splits, default=DEFAULT_SPLITS, metavar="LIST", help="comma-separated split seeds (default: 0,1,2,3,4)")
    parser.add_argument("--final", action="store_true", help="--set eval only: confirm that these are the final segmenters (the eval images are looked at once)")
    parser.add_argument("--again", action="store_true", help="--set eval only: measure a segmenter again that was already measured")
    parser.add_argument("--seg-dir", type=Path, default=None, metavar="DIR", help="the S-seg runs folder to read (default: _work/runs/S-seg)")
    parser.add_argument("--figure", type=Path, default=None, metavar="PATH", help="where to write the examples figure (default: see seg-eval in 40_segmentation.ipynb)")


@dataclass(frozen=True)
class Command:
    name: str
    help: str
    handler: Callable[[argparse.Namespace], int]
    configure: Callable[[argparse.ArgumentParser], None] | None = None
    description: str | None = None  # long help of the command (line breaks kept); default: ``help``


COMMANDS = (
    Command(
        "download-weights",
        "fetch the torchvision pretrained weights into <data-root>/_work/weights/ (idempotent)",
        cmd_download_weights,
    ),
    Command(
        "prepare",
        "index, cache, near-duplicate audit, splits, tables and figures (idempotent)",
        cmd_prepare,
        _configure_prepare,
    ),
    Command("run", "train and evaluate one experiment over the given splits", cmd_run, _configure_run),
    Command(
        "choose-lr",
        "record the learning rate of a backbone from its split-0 sweep (T3.4 rule) in choices.json (or verify it against the recorded one)",
        cmd_choose_lr,
        _configure_choose_lr,
    ),
    Command(
        "choose-composites",
        "record the composites per epoch of Method A from its split-0 sweep (validation only) in choices.json",
        cmd_choose_composites,
        _configure_choose_composites,
    ),
    Command(
        "gate",
        "recompute decision gate g0 (input mode), g1 (backbone) or g2 (recommended method) from the runs and record it in choices.json "
        "(or verify it against the recorded one)",
        cmd_gate,
        _configure_gate,
    ),
    Command(
        "seg-eval",
        "segmenter metrics on the annotated field images: --set dev (development) or --set eval --final (reporting, once)",
        cmd_seg_eval,
        _configure_seg_eval,
    ),
    Command("summarize", "aggregate all runs into tables, figures, failure cases and findings", cmd_summarize),
    Command(
        "benchmark",
        "time inference per image of the three backbones on this device and write _work/results/inference_benchmark.json (summarize reads it)",
        cmd_benchmark,
    ),
    Command(
        "gradcam",
        "Grad-CAM of the plain model and Method A: figures of the success and failure images, and the shortcut check on the annotated field images",
        cmd_gradcam,
        _configure_gradcam,
    ),
    Command("predict", "classify a single image and print class probabilities", cmd_predict, _configure_predict),
    Command("smoke", "tiny end-to-end check: imports, device, data path, one tiny experiment and predict", cmd_smoke),
    Command(
        "reproduce",
        "the whole reported chain (weights, prepare, every experiment, gates, tables), or with --recommended just the recommended model",
        cmd_reproduce,
        _configure_reproduce,
        REPRODUCE_DESCRIPTION,
    ),
)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog=Path(__file__).name,
        description="CSCI935 rice leaf disease classification - reproduces every reported number.",
        epilog="Global flags may be placed before or after the command.",
    )
    add_global_flags(parser, is_subcommand=False)
    subparsers = parser.add_subparsers(dest="command", required=True, metavar="COMMAND")
    for command in COMMANDS:
        sub = subparsers.add_parser(command.name, help=command.help, description=command.description or command.help,
                                    formatter_class=argparse.RawDescriptionHelpFormatter if command.description else argparse.HelpFormatter)
        if command.configure is not None:
            command.configure(sub)
        add_global_flags(sub, is_subcommand=True)
        sub.set_defaults(handler=command.handler)
    return parser

In [ ]:
#| export
def run_cli(argv: list[str] | None = None) -> int:
    """Parse argv and run the chosen command. Returns the process exit code."""
    args = build_parser().parse_args(argv)
    return args.handler(args)


def main() -> None:
    sys.exit(run_cli())